# Experiment 10: Transfer Learning using VGG16 to Classify Images

**Aim:** Re-use a VGG16 network pre-trained on ImageNet to classify a new image dataset (CIFAR-10) with a small amount of data and compute.

**Approach**
1. **Feature extraction:** load VGG16 with ImageNet weights, drop its classifier (`include_top=False`), **freeze** the convolutional base and train only a new classification head.
2. **Fine-tuning:** unfreeze the last convolutional block (`block5`) and continue training with a very small learning rate.
3. Compare with a small CNN trained **from scratch** on the same data.

**Dataset:** CIFAR-10 subset (5,000 train / 1,000 test images). Images are up-sampled from 32x32 to 64x64 (VGG16 accepts >= 32x32 input) to give the pretrained filters more spatial resolution.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

tf.random.set_seed(1); np.random.seed(1)
CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
N_TRAIN, N_TEST, IMG = 5000, 1000, 64
EPOCHS_HEAD, EPOCHS_FT, EPOCHS_SCRATCH = 6, 3, 6
print("TensorFlow", tf.__version__)

## 1. Load a balanced CIFAR-10 subset

In [ ]:
(x_tr, y_tr), (x_te, y_te) = keras.datasets.cifar10.load_data()
y_tr, y_te = y_tr.ravel(), y_te.ravel()

def balanced_subset(x, y, n):
    idx = np.concatenate([np.where(y == c)[0][: n // 10] for c in range(10)])
    np.random.shuffle(idx); return x[idx], y[idx]

x_tr, y_tr = balanced_subset(x_tr, y_tr, N_TRAIN)
x_te, y_te = balanced_subset(x_te, y_te, N_TEST)
print(x_tr.shape, x_te.shape, np.bincount(y_tr))

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for ax, i in zip(axes, range(10)):
    ax.imshow(x_tr[i]); ax.set_title(CLASSES[y_tr[i]], fontsize=8); ax.axis("off")
plt.show()

## 2. Input pipeline
Images are resized to 64x64 and passed through VGG16's own `preprocess_input` (RGB->BGR, ImageNet mean subtraction). Light augmentation (horizontal flip) is applied to the training set only.

In [ ]:
def make_ds(x, y, train):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if train: ds = ds.shuffle(len(x), seed=1)
    def prep(img, lbl):
        img = tf.image.resize(tf.cast(img, tf.float32), (IMG, IMG))
        if train: img = tf.image.random_flip_left_right(img)
        return preprocess_input(img), lbl
    return ds.map(prep, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)

train_ds, test_ds = make_ds(x_tr, y_tr, True), make_ds(x_te, y_te, False)

## 3. Load pre-trained VGG16 (ImageNet weights, without the top classifier)

In [ ]:
base = VGG16(weights="imagenet", include_top=False, input_shape=(IMG, IMG, 3))
base.trainable = False                       # freeze all convolutional layers
print("Base layers:", len(base.layers), "| trainable params:", sum(int(np.prod(w.shape)) for w in base.trainable_weights))
base.summary(line_length=90)

## 4. Add a new classification head (Stage 1: feature extraction)

In [ ]:
def build_transfer():
    inp = keras.Input((IMG, IMG, 3))
    x = base(inp, training=False)            # keep BatchNorm/inference behaviour fixed
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inp, out, name="VGG16_transfer")

model = build_transfer()
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
h1 = model.fit(train_ds, validation_data=test_ds, epochs=EPOCHS_HEAD, verbose=2)

## 5. Stage 2: fine-tune the last convolutional block (`block5`)
Only `block5_*` layers are unfrozen and a ~100x smaller learning rate is used, so the pretrained features are gently adapted instead of destroyed.

In [ ]:
base.trainable = True
for layer in base.layers:
    layer.trainable = layer.name.startswith("block5")
print("Trainable layers:", [l.name for l in base.layers if l.trainable])

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=test_ds, epochs=EPOCHS_FT, verbose=2)

## 6. Baseline: small CNN trained from scratch

In [ ]:
scratch = keras.Sequential([
    layers.Input((IMG, IMG, 3)),
    layers.Conv2D(32, 3, activation="relu", padding="same"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu", padding="same"), layers.MaxPooling2D(),
    layers.Flatten(), layers.Dense(256, activation="relu"), layers.Dropout(0.5), layers.Dense(10, activation="softmax"),
], name="scratch_cnn")
scratch.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hs = scratch.fit(train_ds, validation_data=test_ds, epochs=EPOCHS_SCRATCH, verbose=2)

## 7. Results

In [ ]:
acc_h = h1.history["val_accuracy"] + h2.history["val_accuracy"]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(range(1, len(acc_h)+1), acc_h, "o-", label="VGG16 transfer")
ax[0].axvline(EPOCHS_HEAD + 0.5, color="gray", ls="--"); ax[0].text(EPOCHS_HEAD + 0.6, min(acc_h), "fine-tune ->", fontsize=8)
ax[0].plot(range(1, EPOCHS_SCRATCH+1), hs.history["val_accuracy"], "s-", label="CNN from scratch")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("validation accuracy"); ax[0].legend(); ax[0].set_title("Validation accuracy")

pred = model.predict(test_ds, verbose=0).argmax(1)
ConfusionMatrixDisplay(confusion_matrix(y_te, pred), display_labels=CLASSES).plot(ax=ax[1], cmap="Blues", colorbar=False, xticks_rotation=60)
ax[1].set_title("VGG16 transfer - confusion matrix"); plt.tight_layout(); plt.show()

print(f"VGG16 transfer (head only)  : {h1.history['val_accuracy'][-1]*100:.2f}%")
print(f"VGG16 transfer (fine-tuned) : {h2.history['val_accuracy'][-1]*100:.2f}%")
print(f"CNN from scratch            : {hs.history['val_accuracy'][-1]*100:.2f}%")

## 8. Sample predictions

In [ ]:
imgs, lbls = x_te[:16], y_te[:16]
p = model.predict(preprocess_input(tf.image.resize(imgs.astype("float32"), (IMG, IMG)).numpy()), verbose=0)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, im, l, pr in zip(axes.ravel(), imgs, lbls, p):
    ax.imshow(im); ax.axis("off")
    ax.set_title(f"{CLASSES[pr.argmax()]}\n({pr.max():.2f})", fontsize=9, color="green" if pr.argmax()==l else "red")
plt.show()

## Conclusion
Re-using the ImageNet-trained VGG16 convolutional base gives a strong accuracy with only 5,000 training images and a few epochs, clearly beating a CNN trained from scratch under the same budget. Fine-tuning the last block adapts the high-level features to the new dataset and gives a further gain. This shows why transfer learning is the default when labelled data or compute is limited.